# City-Level Analysis — Event Duration and Inter-Annual Variability

Analyses compound cold-dry (CD) and cold-wet (CW) event characteristics at
five UK cities: Belfast, Newcastle, Glasgow, London, Plymouth.

**Prerequisites:** Per-city event date arrays must exist in `data/arrays/`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
from collections import defaultdict
from matplotlib.ticker import FuncFormatter

# ── User configuration ──────────────────────────────────────────────────────
data_arrays_dir = "data/arrays"
fig_dir         = "figures"
STUDY_START     = 1960
STUDY_END       = 2022
CITIES          = ["belfast", "newcastle", "glasgow", "london", "plymouth"]
CITY_COLOURS    = {
    "belfast":   "#4848EB",
    "newcastle": "#B885E6",
    "glasgow":   "#777484",
    "london":    "#257E77",
    "plymouth":  "#73E2EA",
}
os.makedirs(fig_dir, exist_ok=True)

def format_as_integer(x, pos):
    return f"{int(x)}"


## 1. Helper: load and process city data

In [ ]:
def load_and_process_city(city, event_type):
    """
    Load event dates for one city and return event-length list and
    per-year counts (NaN for years with no events).

    Parameters
    ----------
    city : str  e.g. 'belfast'
    event_type : str  'CD' or 'CW'

    Returns
    -------
    event_lengths : list[int]
    complete_events_per_year : dict {year: int or np.nan}
    """
    fname = os.path.join(data_arrays_dir,
                         f"{event_type}_dates_filtered_{city}.npy")
    raw_data = np.load(fname, allow_pickle=True)
    raw_data[:, 0] = np.array([np.datetime64(d) for d in raw_data[:, 0]])
    dates = np.sort(raw_data[:, 0])

    # Consecutive-run lengths
    run, lengths = 0, []
    for i in range(len(dates) - 1):
        if dates[i + 1] - dates[i] == timedelta(days=1):
            run += 1
        else:
            lengths.append(run + 1)
            run = 0
    lengths.append(run + 1)

    # Per-year counts
    per_year = defaultdict(int)
    for d in dates:
        per_year[d.year] += 1
    complete = {yr: per_year.get(yr, np.nan)
                for yr in range(STUDY_START, STUDY_END + 1)}
    return lengths, complete


## 2. Event-length histograms

In [ ]:
for event_type in ["CD", "CW"]:
    fig, axes = plt.subplots(1, len(CITIES), figsize=(18, 4), sharey=True)
    fig.suptitle(f"{event_type} — Event Length Distributions", fontsize=14)
    for ax, city in zip(axes, CITIES):
        lengths, _ = load_and_process_city(city, event_type)
        multi = sum(1 for l in lengths if l > 1)
        ax.hist(lengths, bins=range(1, max(lengths) + 2), edgecolor="black")
        ax.set_title(city.capitalize(), fontsize=11)
        ax.set_xlabel("Event Length (days)")
        ax.set_ylim(0, 175)
        ax.text(0.95, 0.95, f"Multi-day:\n{multi}",
                transform=ax.transAxes, ha="right", va="top", fontsize=9)
    axes[0].set_ylabel("Frequency")
    plt.tight_layout()
    fig.savefig(os.path.join(fig_dir, f"{event_type}_event_length_histograms.png"), dpi=300)
    plt.show()


## 3. Inter-annual variability (combined CD and CW figure)

In [ ]:
bar_width, offset = 0.15, 0.20
offsets = [-2, -1, 0, 1, 2]

fig, ax = plt.subplots(figsize=(15, 7), nrows=2)
for row, event_type in enumerate(["CD", "CW"]):
    for city, off in zip(CITIES, offsets):
        _, per_year = load_and_process_city(city, event_type)
        years  = list(per_year.keys())
        counts = list(per_year.values())
        xs = [yr + off * offset for yr in years]
        ax[row].bar(xs, counts, width=bar_width,
                    label=city.capitalize(), color=CITY_COLOURS[city])
    ax[row].set_ylabel("Day Counts", fontsize=14)
    ax[row].grid(which="major", linestyle="dashed", alpha=0.3)
    ax[row].text(-0.06, 1.05, "AB"[row],
                 transform=ax[row].transAxes, fontsize=20,
                 fontweight="bold", va="top", ha="right")

ax[0].set_ylim(0, 50);  ax[0].legend(fontsize=12);  ax[0].set_xticklabels([])
ax[0].yaxis.set_major_formatter(FuncFormatter(format_as_integer))
ax[1].set_ylim(0, 6);   ax[1].set_yticks(np.arange(0, 6, 1))
ax[1].yaxis.set_major_formatter(FuncFormatter(format_as_integer))
ax[1].tick_params(axis="x", labelsize=14)
plt.xlabel("Year", fontsize=15)
fig.tight_layout(pad=3.0)
fig.savefig(os.path.join(fig_dir, "CD_CW_days_per_year_cities.png"), dpi=300, facecolor="white")
plt.show()
